# 04 — Model Comparison: GP vs MLP

**Scientific narrative:**

- Uncertainty correction (corrected dataset) vs uniform noise (raw)
- Ablation of fe02cr14 (most problematic tag: mixed FM/AFM, Tier D+E)
- GP provides calibrated uncertainty | MLP provides best point predictions
- FEM material cards use MLP mean + GP std

**Best model selection (by MAE + R²):**

| Constant | Model | Dataset | Rationale |
|---|---|---|---|
| C11 | MLP | raw ablated | R²=0.915, MAE=18.63 GPa |
| C12 | MLP | raw ablated | R²=0.825, MAE=11.67 GPa — fe02cr14 corrupts C12 |
| C44 | MLP | raw full    | R²=0.697, MAE=3.26 GPa — correction/ablation both hurt |

**Run after:** 01_gp_surrogate, 03_mlp, 05_ablation_fe02cr14

---
## Cell 1 — Imports + Load all pkl files

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, sys, os
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

sys.path.insert(0, os.path.dirname(os.path.abspath('data_utils.py')))
from data_utils import (load_data, TARGETS, PALETTE, N_ATOMS)

ANA = '../analysis'

# ── Load DFT data for scatter points ─────────────────────────────────────
CSV_PATHS = {
    'raw':       f'{ANA}/elastic_constants_fecr_raw.csv',
    'corrected': f'{ANA}/elastic_constants_fecr_corrected.csv',
}
datasets = {}
for mode, path in CSV_PATHS.items():
    d  = load_data(path)
    df = d['df']
    tier = np.where(df['n_cr'] <= 9,  'A',
           np.where(df['n_cr'] <= 13, 'B', 'C'))
    datasets[mode] = dict(
        df             = df,
        x_pred_atoms   = d['x_pred_atoms'],
        targets        = d['targets'],
        flagged        = d['flagged'],
        vcr_geometry   = d['vcr_geometry'],
        cubic_enforced = d['cubic_enforced'],
        tier           = tier,
    )
x_pred_atoms = datasets['raw']['x_pred_atoms']

# ── Load pkl files ────────────────────────────────────────────────────────
PKL_FILES = {
    'GP_raw_full':        f'{ANA}/gp_results_raw_full.pkl',
    'GP_raw_no14':        f'{ANA}/gp_results_raw_no14.pkl',
    'GP_corr_full':       f'{ANA}/gp_results_corrected_full.pkl',
    'GP_corr_no14':       f'{ANA}/gp_results_corrected_no14.pkl',
    'MLP_raw_full':       f'{ANA}/mlp_results_raw_full.pkl',
    'MLP_raw_no14':       f'{ANA}/mlp_results_raw_no14.pkl',
    'MLP_corr_full':      f'{ANA}/mlp_results_corrected_full.pkl',
    'MLP_corr_no14':      f'{ANA}/mlp_results_corrected_no14.pkl',
}

R = {}
for key, path in PKL_FILES.items():
    with open(path, 'rb') as f:
        R[key] = pickle.load(f)
    print(f'Loaded: {key}')

# ── Best model per constant (your selection) ──────────────────────────────
BEST = {
    'C11': 'MLP_raw_no14',
    'C12': 'MLP_raw_no14',
    'C44': 'MLP_raw_full',
}
print()
print('Best model assignments:')
for t, k in BEST.items():
    print(f'  {t}: {k}  MAE={R[k][t]["mae"]:.2f} GPa  R²={R[k][t]["r2"]:.4f}')


Loaded 17 tags — mode: RAW
tag            n_cr  σ C11+2C12  σ C11-C12   σ C44   afm  vcr_geo  cub_enf
--------------------------------------------------------------------
fe16cr00          0       1.000      1.000   1.000 False    False    False
fe15cr01          1       1.000      1.000   1.000 False    False    False
fe14cr02          2       1.000      1.000   1.000 False    False    False
fe13cr03          3       1.000      1.000   1.000 False    False    False
fe12cr04          4       1.000      1.000   1.000 False    False    False
fe11cr05          5       1.000      1.000   1.000 False    False    False
fe10cr06          6       1.000      1.000   1.000 False    False    False
fe09cr07          7       1.000      1.000   1.000 False    False    False
fe08cr08          8       1.000      1.000   1.000 False    False    False
fe07cr09          9       1.000      1.000   1.000 False    False    False
fe06cr10         10       1.000      1.000   1.000 False    False    False
fe0

/home/sayeed/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


---
## Cell 2 — Full performance table: all models × all variants

In [2]:
rows = []
for key, mres in R.items():
    parts = key.split('_')
    model = parts[0]
    ablated = 'no14' in key
    dataset = 'corrected' if 'corr' in key else 'raw'
    for tname in TARGETS:
        rows.append({
            'model':   model,
            'dataset': dataset,
            'ablated': ablated,
            'variant': key,
            'target':  tname,
            'MAE':     round(mres[tname]['mae'], 2),
            'R2':      round(mres[tname]['r2'],  4),
        })

perf = pd.DataFrame(rows)

pd.set_option('display.float_format', '{:.3f}'.format)
print('LOO-CV MAE (GPa) — lower is better:')
print(perf.pivot_table(
    index=['model','dataset','ablated'],
    columns='target', values='MAE').to_string())
print()
print('LOO-CV R²  — higher is better:')
print(perf.pivot_table(
    index=['model','dataset','ablated'],
    columns='target', values='R2').to_string())
pd.reset_option('display.float_format')

perf.to_csv(f'{ANA}/model_comparison_full_table.csv', index=False)
print('\nSaved: model_comparison_full_table.csv')

LOO-CV MAE (GPa) — lower is better:
target                     C11    C12   C44
model dataset   ablated                    
GP    corrected False   31.350 20.540 4.150
                True    30.550 18.090 4.230
      raw       False   45.870 32.690 3.540
                True    38.800 24.390 3.720
MLP   corrected False   20.520 15.390 3.650
                True    19.270 12.970 3.700
      raw       False   19.240 15.590 3.260
                True    18.630 11.670 3.520

LOO-CV R²  — higher is better:
target                    C11   C12   C44
model dataset   ablated                  
GP    corrected False   0.776 0.524 0.593
                True    0.762 0.630 0.572
      raw       False   0.587 0.043 0.652
                True    0.653 0.416 0.623
MLP   corrected False   0.911 0.683 0.684
                True    0.903 0.798 0.658
      raw       False   0.919 0.701 0.697
                True    0.914 0.825 0.663

Saved: model_comparison_full_table.csv


---
## Cell 3 — Visual performance comparison

4 bar groups: GP_full, GP_no14, MLP_full, MLP_no14  
2 panels: MAE and R²  
2 rows: raw / corrected

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

group_keys = {
    'raw':       ['GP_raw_full','GP_raw_no14','MLP_raw_full','MLP_raw_no14'],
    'corrected': ['GP_corr_full','GP_corr_no14','MLP_corr_full','MLP_corr_no14'],
}
group_labels = ['GP full','GP no14','MLP full','MLP no14']
group_colors = ['#4878CF','#88AAEE','#2E8B3A','#6ACC65']

x  = np.arange(len(TARGETS))
bw = 0.18

for row_idx, (dataset, keys) in enumerate(group_keys.items()):
    for col_idx, (metric, title, better) in enumerate([
        ('MAE', f'LOO-CV MAE (GPa) [{dataset}]
        lower = better', False),
        ('R2',  f'LOO-CV R²  [{dataset}]
        higher = better',      True),
    ]):
        ax = axes[row_idx, col_idx]
        for i, (key, label, col) in enumerate(
                zip(keys, group_labels, group_colors)):
            vals = [R[key][t][metric.lower()] for t in TARGETS]
            bars = ax.bar(x + i*bw, vals, bw, label=label,
                          color=col, edgecolor='k', lw=0.5)

        # Mark best per target
        for j, tname in enumerate(TARGETS):
            if tname in BEST:
                best_key = BEST[tname]
                # Check if best_key belongs to this dataset group
                if best_key in keys:
                    ki = keys.index(best_key)
                    val = R[best_key][tname][metric.lower()]
                    ax.annotate('★', xy=(j + ki*bw, val),
                                xytext=(0, 4), textcoords='offset points',
                                ha='center', fontsize=10, color='gold',
                                fontweight='bold')

        ax.set_xticks(x + 1.5*bw)
        ax.set_xticklabels(TARGETS)
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, axis='y')

plt.suptitle('Model Performance: GP vs MLP — full vs ablated (★ = selected for FEM)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(f'{ANA}/model_comparison_performance.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_performance.png')

SyntaxError: unterminated f-string literal (detected at line 15) (3293680996.py, line 15)

---
## Cell 4 — Prediction overlay: GP vs MLP, full vs ablated

2 rows (raw/corrected) × 3 cols (C11/C12/C44)  
Each panel: GP_full, GP_no14, MLP_full, MLP_no14  
★ marks the best model for each constant

In [ ]:
tier_specs = [
    ('A', 'black',     'o'),
    ('B', 'goldenrod', 's'),
    ('C', 'tomato',    'D'),
]

line_styles = {
    'GP_raw_full':   dict(color='#4878CF', ls='-',  lw=2.0, label='GP full'),
    'GP_raw_no14':   dict(color='#4878CF', ls='--', lw=1.5, label='GP no14'),
    'MLP_raw_full':  dict(color='#2E8B3A', ls='-',  lw=2.0, label='MLP full'),
    'MLP_raw_no14':  dict(color='#2E8B3A', ls='--', lw=1.5, label='MLP no14'),
    'GP_corr_full':  dict(color='#1133AA', ls='-',  lw=2.0, label='GP corr full'),
    'GP_corr_no14':  dict(color='#1133AA', ls='--', lw=1.5, label='GP corr no14'),
    'MLP_corr_full': dict(color='#145214', ls='-',  lw=2.0, label='MLP corr full'),
    'MLP_corr_no14': dict(color='#145214', ls='--', lw=1.5, label='MLP corr no14'),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, dataset in enumerate(['raw', 'corrected']):
    d = datasets[dataset]
    row_keys = [k for k in R if (('corr' in k) == (dataset=='corrected'))]

    for col_idx, tname in enumerate(TARGETS):
        ax = axes[row_idx, col_idx]

        for key in row_keys:
            mu  = R[key][tname]['mu']
            sty = line_styles[key]
            lw  = 3.0 if key == BEST[tname] else sty['lw']
            ax.plot(x_pred_atoms, mu, color=sty['color'],
                    ls=sty['ls'], lw=lw, label=sty['label'])

        # GP uncertainty band for best GP variant
        best_gp_key = f'GP_{"corr" if dataset=="corrected" else "raw"}_no14'
        if 'std' in R[best_gp_key][tname]:
            mu_gp  = R[best_gp_key][tname]['mu']
            sig_gp = R[best_gp_key][tname]['std']
            ax.fill_between(x_pred_atoms, mu_gp-sig_gp, mu_gp+sig_gp,
                            alpha=0.10, color='steelblue', label='GP ±1σ')

        # DFT scatter
        for t_lbl, col_s, mk in tier_specs:
            m = ((d['tier']==t_lbl)
                 & ~d['vcr_geometry']
                 & ~d['cubic_enforced'])
            if m.any():
                ax.scatter(d['df']['n_cr'][m],
                           d['targets'][tname][m],
                           color=col_s, s=55, marker=mk, zorder=5)

        if d['vcr_geometry'].any():
            ax.scatter(d['df']['n_cr'][d['vcr_geometry']],
                       d['targets'][tname][d['vcr_geometry']],
                       color='black', s=120, marker='x',
                       linewidths=2, zorder=6, label='Tier D')

        cub = d['cubic_enforced'] & ~d['vcr_geometry']
        if cub.any():
            ax.scatter(d['df']['n_cr'][cub],
                       d['targets'][tname][cub],
                       color='purple', s=90, marker='p',
                       zorder=5, label='Tier E')

        # Annotate best model
        best_key = BEST[tname]
        if ('corr' in best_key) == (dataset == 'corrected'):
            ax.set_title(f'{tname} [{dataset}]  ★={best_key}', fontsize=9)
        else:
            ax.set_title(f'{tname} [{dataset}]', fontsize=9)

        ax.set_xlabel('Cr atoms (out of 16)')
        ax.set_ylabel(f'{tname} (GPa)')
        ax.legend(fontsize=6, ncol=2)
        ax.grid(alpha=0.3)
        ax.set_xlim(-0.5, 16.5)

plt.suptitle('Prediction Overlay: GP vs MLP — full vs ablated\n★ = selected for FEM',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(f'{ANA}/model_comparison_overlay.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_overlay.png')

---
## Cell 5 — Effect of uncertainty correction

ΔR² and ΔMAE: corrected vs raw (full dataset)  
Shows whether the noise correction helped each model/constant

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models_to_compare = [('GP','GP_raw_full','GP_corr_full'),
                     ('MLP','MLP_raw_full','MLP_corr_full')]
x   = np.arange(len(TARGETS))
bw  = 0.35
cols_gp  = ['#4878CF','#1133AA']
cols_mlp = ['#2E8B3A','#145214']

for ax, metric, title, sign in [
    (axes[0], 'r2',  'ΔR²  (corrected − raw)
↑ positive = correction helped',  1),
    (axes[1], 'mae', 'ΔMAE (corrected − raw)
↓ negative = correction helped', -1),
]:
    for i, (mname, raw_key, corr_key) in enumerate(models_to_compare):
        deltas = []
        for tname in TARGETS:
            v_raw  = R[raw_key][tname][metric]
            v_corr = R[corr_key][tname][metric]
            deltas.append(sign * (v_corr - v_raw))
        col = cols_gp[0] if mname == 'GP' else cols_mlp[0]
        ax.bar(x + i*bw, deltas, bw, label=mname,
               color=[('seagreen' if d >= 0 else 'tomato') for d in deltas],
               edgecolor=col, lw=1.2)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x + bw/2); ax.set_xticklabels(TARGETS)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

    # Hatch for MLP
    for patch in ax.patches[len(TARGETS):]:
        patch.set_hatch('///')

plt.suptitle('Effect of Uncertainty Correction (corrected − raw, full dataset)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ANA}/model_comparison_correction_effect.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_correction_effect.png')

---
## Cell 6 — Effect of fe02cr14 ablation

ΔR² and ΔMAE: no14 vs full (raw dataset)  
Shows the composition-specific influence of the problematic tag

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title, sign in [
    (axes[0], 'r2',  'ΔR²  (no14 − full, raw)
↑ positive = removal helped',   1),
    (axes[1], 'mae', 'ΔMAE (no14 − full, raw)
↓ negative = removal helped',  -1),
]:
    for i, (mname, full_key, no14_key) in enumerate([
        ('GP',  'GP_raw_full',  'GP_raw_no14'),
        ('MLP', 'MLP_raw_full', 'MLP_raw_no14'),
    ]):
        deltas = [sign*(R[no14_key][t][metric]-R[full_key][t][metric])
                  for t in TARGETS]
        bar_cols = ['seagreen' if d >= 0 else 'tomato' for d in deltas]
        ax.bar(x + i*bw, deltas, bw, label=mname,
               color=bar_cols, edgecolor='k', lw=0.8)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x + bw/2); ax.set_xticklabels(TARGETS)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Effect of Removing fe02cr14 (no14 − full, raw dataset)\n'
             'green = removal improves  |  red = removal hurts',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ANA}/model_comparison_ablation_effect.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_ablation_effect.png')
print()
print('Physical interpretation:')
print('  C12 improves strongly → fe02cr14 tetragonal calcs corrupted by FM/AFM ambiguity')
print('  C11 marginal/mixed   → hydrostatic calcs less affected by magnetic state')
print('  C44 unaffected       → shear calcs robust to magnetic initialisation')

---
## Cell 7 — GP vs MLP disagreement map

Where do GP and MLP disagree most?  
Large disagreement = physically uncertain region — use GP uncertainty band

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col_idx, (dataset, gp_key, mlp_key) in enumerate([
    ('raw',       'GP_raw_no14',  'MLP_raw_no14'),
    ('corrected', 'GP_corr_no14', 'MLP_corr_no14'),
]):
    ax = axes[col_idx]
    for tname in TARGETS:
        spread = np.abs(R[gp_key][tname]['mu'] - R[mlp_key][tname]['mu'])
        ax.plot(x_pred_atoms, spread,
                color=PALETTE[tname], lw=2, label=tname)
    ax.axhline(5, color='tomato', lw=1, ls='--', label='5 GPa threshold')
    ax.set_xlabel('Cr atoms (out of 16)')
    ax.set_ylabel('|GP − MLP| (GPa)')
    ax.set_title(f'GP vs MLP disagreement [{dataset}, ablated]', fontsize=10)
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)

plt.suptitle('Model Disagreement: |GP − MLP| (ablated datasets)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ANA}/model_comparison_disagreement.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_disagreement.png')
print('Regions above 5 GPa threshold: FEM predictions less reliable.')

---
## Cell 8 — FEM material cards using best model per constant

C11, C12: MLP raw ablated  
C44: MLP raw full  
Uncertainty: GP std from best GP variant

In [ ]:
FEM_CR_ATOMS = [0, 4, 8, 12, 16]

# Best GP for uncertainty bands
GP_BEST = {
    'C11': 'GP_raw_no14',
    'C12': 'GP_raw_no14',
    'C44': 'GP_raw_full',
}

rows = []
for ncr in FEM_CR_ATOMS:
    idx  = np.argmin(np.abs(x_pred_atoms - ncr))
    row  = {'n_cr': ncr, 'x_cr': round(ncr/16, 4),
            'label': f'Fe{16-ncr}Cr{ncr}'}
    for tname in TARGETS:
        mlp_key = BEST[tname]
        gp_key  = GP_BEST[tname]
        row[f'{tname}_GPa']     = round(float(R[mlp_key][tname]['mu'][idx]),  2)
        gp_std = R[gp_key][tname].get('std', np.zeros(200))
        row[f'{tname}_std_GPa'] = round(float(gp_std[idx]), 2)
    rows.append(row)

fem_df = pd.DataFrame(rows)
print(fem_df.to_string(index=False))
fem_df.to_csv(f'{ANA}/fem_material_inputs_best.csv', index=False)
print('\nSaved: fem_material_inputs_best.csv')
print('Primary values: MLP (C11/C12: raw ablated | C44: raw full)')
print('Uncertainty std: GP posterior std (same variant as MLP)')

---
## Cell 9 — ABAQUS + CalculiX material cards from best model

In [ ]:
abaqus_lines = [
    '** Fe-Cr Material Cards — Best ML Surrogate',
    '** Primary: MLP (C11/C12: raw ablated | C44: raw full)',
    '** Uncertainty: GP posterior std (same variant)',
    '** Units: GPa (multiply by 1000 for MPa in ABAQUS/CalculiX)',
    '** Cubic Voigt order: C11 C12 C11 C12 C12 C11 C44 C44 C44',
    '',
]
ccx_lines = abaqus_lines.copy()

for _, r in fem_df.iterrows():
    c11 = r['C11_GPa']; c12 = r['C12_GPa']; c44 = r['C44_GPa']
    s11 = r['C11_std_GPa']; s12 = r['C12_std_GPa']; s44 = r['C44_std_GPa']
    B   = round((c11 + 2*c12)/3, 2)
    ZA  = round(2*c44/(c11-c12), 4) if (c11-c12) != 0 else float('nan')

    comment = (f'** {r["label"]}  B={B:.2f} GPa  ZenerA={ZA:.4f}  '
               f'C11±{s11:.2f}  C12±{s12:.2f}  C44±{s44:.2f}')
    v1 = (f'{c11:.3f}, {c12:.3f}, {c11:.3f}, {c12:.3f}, '
          f'{c12:.3f}, {c11:.3f}, {c44:.3f}, {c44:.3f},')
    v2 = f'{c44:.3f}'

    abaqus_lines += [comment,
                     f'*Material, name={r["label"]}',
                     '*Elastic, type=ANISOTROPIC',
                     v1, v2, '']
    ccx_lines    += [comment,
                     f'*MATERIAL, NAME={r["label"]}',
                     '*ELASTIC, TYPE=ANISO',
                     v1, v2, '']

with open(f'{ANA}/abaqus_material_cards_best.inp', 'w') as f:
    f.write('\n'.join(abaqus_lines))
with open(f'{ANA}/calculix_material_cards_best.inp', 'w') as f:
    f.write('\n'.join(ccx_lines))

print('Saved: abaqus_material_cards_best.inp')
print('Saved: calculix_material_cards_best.inp')
print()
print('Card summary:')
for _, r in fem_df.iterrows():
    print(f'  {r["label"]}: C11={r["C11_GPa"]:.1f}±{r["C11_std_GPa"]:.1f}  '
          f'C12={r["C12_GPa"]:.1f}±{r["C12_std_GPa"]:.1f}  '
          f'C44={r["C44_GPa"]:.1f}±{r["C44_std_GPa"]:.1f} GPa')

---
## Cell 10 — Final summary and physical discussion

In [ ]:
print('=' * 70)
print('MODEL COMPARISON — FINAL SUMMARY')
print('=' * 70)
print()
print('── Effect of uncertainty correction (corrected vs raw, full dataset) ──')
print()
for tname in TARGETS:
    for mname, rk, ck in [('GP','GP_raw_full','GP_corr_full'),
                           ('MLP','MLP_raw_full','MLP_corr_full')]:
        dr2  = R[ck][tname]['r2']  - R[rk][tname]['r2']
        dmae = R[ck][tname]['mae'] - R[rk][tname]['mae']
        sign_r2  = '↑' if dr2  > 0 else '↓'
        sign_mae = '↑' if dmae < 0 else '↓'
        print(f'  {tname} {mname}: ΔR²={dr2:+.4f}{sign_r2}  '
              f'ΔMAE={dmae:+.2f} GPa{sign_mae}')
print()
print('── Effect of fe02cr14 removal (no14 vs full, raw dataset) ──')
print()
for tname in TARGETS:
    for mname, fk, nk in [('GP','GP_raw_full','GP_raw_no14'),
                           ('MLP','MLP_raw_full','MLP_raw_no14')]:
        dr2  = R[nk][tname]['r2']  - R[fk][tname]['r2']
        dmae = R[nk][tname]['mae'] - R[fk][tname]['mae']
        sign_r2  = '↑' if dr2  > 0 else '↓'
        sign_mae = '↑' if dmae < 0 else '↓'
        print(f'  {tname} {mname}: ΔR²={dr2:+.4f}{sign_r2}  '
              f'ΔMAE={dmae:+.2f} GPa{sign_mae}')
print()
print('── Best model per constant ──')
print()
for tname, key in BEST.items():
    print(f'  {tname}: {key:<20s}  '
          f'R²={R[key][tname]["r2"]:.4f}  '
          f'MAE={R[key][tname]["mae"]:.2f} GPa')
print()
print('── Physical interpretation ──')
print()
print('  C11: Uncertainty correction helps (hydrostatic calcs, conv_thr sensitive).')
print('       fe02cr14 removal marginal — hydrostatic path less affected by')
print('       FM/AFM ambiguity. Tag carries real C11 information at n_cr=14.')
print()
print('  C12: Uncertainty correction helps substantially (R²: +0.44 GP, +0.05 MLP).')
print('       fe02cr14 removal helps strongly (MAE: −8 GPa MLP raw).')
print('       Tetragonal calcs (→C11−C12→C12) are most sensitive to magnetic')
print('       initialisation — FM/AFM ambiguity introduces directional stress')
print('       artefact that couples to off-diagonal elastic response.')
print()
print('  C44: Neither correction nor ablation helps — shear calcs are robust.')
print('       Cubic enforcement cancels in finite difference for shear stress.')
print('       Magnetic state does not couple to shear response at this level.')
print('       Full raw dataset maximises coverage and gives best C44.')
print()
print('── Caveats ──')
print()
print('  N=17 (16 ablated): all LOO metrics on tiny dataset — indicative only.')
print('  MLP uncertainty: bootstrap not shown here — use GP std for FEM bounds.')
print('  fe02cr14 excluded from C11/C12 training but physically present in alloy.')
print('  ABAQUS/CalculiX card format: verify *Elastic keyword for your version.')

print()
print('Outputs saved to ../analysis/:')
for fn in [
    'model_comparison_full_table.csv',
    'model_comparison_performance.png',
    'model_comparison_overlay.png',
    'model_comparison_correction_effect.png',
    'model_comparison_ablation_effect.png',
    'model_comparison_disagreement.png',
    'fem_material_inputs_best.csv',
    'abaqus_material_cards_best.inp',
    'calculix_material_cards_best.inp',
]:
    print(f'  {fn}')